In [ ]:
# Install required packages for dataset download and ONNX export utilities
!pip install -q kaggle
!pip install onnxscript

import os

# Set Kaggle API credentials (required when running in Colab)
os.environ['KAGGLE_USERNAME'] = "INSERT_USERNAME_HERE"
os.environ['KAGGLE_KEY'] = "INSERT_KEY_HERE"

# Download and extract the PlantVillage dataset to a local directory
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d dataset
print("Dataset successfully downloaded and unzipped!")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:20<00:00, 108MB/s]

Dataset successfully downloaded and unzipped!


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms as T
from torch.utils.data import DataLoader, random_split

# Apply standard augmentation and normalization for ImageNet-pretrained backbones
image_transformation = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# PlantVillage RGB images are stored in the 'color' subdirectory
data_dir = 'dataset/plantvillage dataset/color'
full_dataset = datasets.ImageFolder(root=data_dir, transform=image_transformation)

# 80/20 Train-Test Split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size])

# Build data loaders for efficient mini-batch training and evaluation
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
class_names = full_dataset.classes

print(f"Success! Found {len(class_names)} classes: {class_names[:3]}...")

Success! Found 38 classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']...


In [ ]:
# Select GPU when available; otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load a pretrained MobileNetV2 model for transfer learning
model = models.mobilenet_v2(weights='DEFAULT')

# Freeze feature extractor weights during initial training
for param in model.parameters():
    param.requires_grad = False

# Replace final classifier layer with task-specific output dimension
num_classes = len(class_names)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)

print(f"Model loaded and sent to {device}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 141MB/s]


Model loaded and sent to cuda


In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Use cross-entropy loss for multi-class classification
criterion = nn.CrossEntropyLoss()
# Configure optimizer for classifier warm-up and adaptive learning-rate scheduling
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# Train for one epoch and return mean training loss
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

# Evaluate model performance on the validation set and return accuracy
def validate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# Warm-up phase: train only the classifier head for rapid task adaptation (3 Epochs)
print("Starting Warm-up Phase...")
for epoch in range(3):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    accuracy = validate(model, val_loader)
    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {accuracy:.2f}%")
    scheduler.step(accuracy)


# Unfreeze upper feature blocks to begin fine-tuning
print("\nUnfreezing top layers for fine-tuning...")
for param in model.features[14:].parameters():
    param.requires_grad = True

# Reinitialize optimizer to include newly trainable parameters
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Fine-tune for 7 epochs to improve generalization
for epoch in range(7): # 7 more epochs for a total of 10
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    accuracy = validate(model, val_loader)
    print(f"Fine-tune Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {accuracy:.2f}%")
    scheduler.step(accuracy)

Starting Warm-up Phase...


Training: 100%|██████████| 1358/1358 [04:25<00:00,  5.11it/s]


Epoch 1 | Loss: 0.6829 | Val Acc: 93.57%


Training: 100%|██████████| 1358/1358 [04:20<00:00,  5.22it/s]


Epoch 2 | Loss: 0.2753 | Val Acc: 94.72%


...

Fine-tune Epoch 5 | Loss: 0.0266 | Val Acc: 99.08%


Training: 100%|██████████| 1358/1358 [04:37<00:00,  4.89it/s]


Fine-tune Epoch 6 | Loss: 0.0224 | Val Acc: 99.15%


Training: 100%|██████████| 1358/1358 [04:39<00:00,  4.86it/s]


Fine-tune Epoch 7 | Loss: 0.0208 | Val Acc: 99.23%


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from matplotlib.colors import LogNorm

model.eval()
all_preds = []
all_labels = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Running evaluation on {device}...")

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())


# Build a confusion matrix to inspect class-level prediction errors
cm = confusion_matrix(all_labels, all_preds)

# Normalize each row to compare per-class performance consistently
cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

plt.figure(figsize=(14, 11))

# Use logarithmic color scaling to expose both common and rare errors
# Values below 0.1% appear nearly white; 100% appears darkest
sns.heatmap(cm_normalized + 1e-9, 
          annot=False, 
          cmap='Blues', 
          norm=LogNorm(vmin=1e-3, vmax=1.0))

plt.title('Log-Scaled Normalized Confusion Matrix', fontsize=22)
plt.xlabel('Predicted Class', fontsize=16)
plt.ylabel('True Class', fontsize=16)

plt.tight_layout()

In [ ]:
import torch
import onnx
import os
from google.colab import files

# Export the trained model to ONNX and bundle parameters into a single artifact
# --- Configuration ---
OUTPUT_PATH = "plant_model.onnx"
INPUT_SHAPE = (1, 3, 224, 224)

# Ensure model is in evaluation mode and moved to CPU for stable export
model.eval()
model.to('cpu') # Exporting on CPU is more stable for single-file bundling
dummy_input = torch.randn(*INPUT_SHAPE)

print("🚀 Exporting directly to ONNX (Legacy Mode)...")

# Perform ONNX export (legacy mode avoids external .data file splitting)
torch.onnx.export(
    model,
    dummy_input,
    OUTPUT_PATH,
    opset_version=12,
    input_names=["input"],
    output_names=["output"],
    do_constant_folding=True,
    export_params=True,
    dynamo=False, 
)

# Merge external tensor data back into one ONNX file if needed
print("📦 Embedding weights into binary...")
onnx_model = onnx.load(OUTPUT_PATH, load_external_data=True)
onnx.save_model(onnx_model, OUTPUT_PATH, save_as_external_data=False)

# Remove temporary external data file and report final model size
if os.path.exists(OUTPUT_PATH + ".data"):
    os.remove(OUTPUT_PATH + ".data")

size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print("-" * 30)
print(f"✅ SUCCESS: {OUTPUT_PATH} created!")
print(f"📊 Final Size: {size_mb:.2f} MB")
print("-" * 30)

# Download the exported model from the Colab environment
files.download(OUTPUT_PATH)